# Photo / PDF Story Video Maker  (no API key needed)

**ZIP of photos, a PDF, or both + one sentence per photo / PDF page  ->  MP4 with voice, camera motion, transitions, music and subtitles.**

* Photo 1 gets sentence 1, photo 2 gets sentence 2, and so on (photos are used in file-name order: `1.jpg, 2.jpg, 10.jpg` ...).
* **2 or 3 pictures for one sentence:** set `IMAGES_PER_SCENE` in cell 3. With `2`, pictures 1-2 go to sentence 1, pictures 3-4 to sentence 2, and so on; the sentence's time is shared between its pictures.
* **PDF:** every page of a PDF counts as one photo, so a 6-page PDF needs 6 sentences (or 12 with `IMAGES_PER_SCENE = 2`). A PDF can be uploaded on its own, or inside the ZIP together with photos.
* Everything is free: the voices come from `edge-tts`, the music is generated here, translation uses `deep-translator`. No Gemini / OpenAI key, no GPU.
* The voice can be changed in cell 3 (and tried out in cell 6).

Run the cells in order. In cell 3 write your sentences, then in cell 4 upload your ZIP / PDF.

## 1. Install

In [ ]:
%pip install -q -U edge-tts imageio-ffmpeg deep-translator pillow numpy pymupdf

## 2. Helpers

In [ ]:
import os, re, io, math, shutil, hashlib, subprocess, asyncio, threading, wave, zipfile, time
import numpy as np
from PIL import Image, ImageOps, ImageFilter

W, H, FPS = 1280, 720, 24
OUTPUT_DIR = "/content/photo_video"
CACHE_DIR  = os.path.join(OUTPUT_DIR, "audio_cache")
PHOTO_DIR  = os.path.join(OUTPUT_DIR, "photos")
for d in (CACHE_DIR, PHOTO_DIR): os.makedirs(d, exist_ok=True)

def find_ffmpeg():
    p = shutil.which("ffmpeg")
    if p: return p
    import imageio_ffmpeg
    return imageio_ffmpeg.get_ffmpeg_exe()
FFMPEG = find_ffmpeg()

def get_duration(path):
    out = subprocess.run([FFMPEG, "-i", path], capture_output=True, text=True).stderr
    m = re.search(r"Duration:\s*(\d+):(\d+):(\d+\.?\d*)", out)
    if not m: raise RuntimeError("Cannot read duration: " + path)
    h, mi, s = m.groups()
    return int(h) * 3600 + int(mi) * 60 + float(s)

def run_async(coro):
    box = {}
    def target():
        try: box["v"] = asyncio.run(coro)
        except Exception as e: box["e"] = e
    t = threading.Thread(target=target); t.start(); t.join()
    if "e" in box: raise box["e"]
    return box.get("v")

BACKUP_VOICE = {"hi-IN-MadhurNeural": "hi-IN-SwaraNeural", "hi-IN-SwaraNeural": "hi-IN-MadhurNeural",
                "mr-IN-ManoharNeural": "mr-IN-AarohiNeural", "mr-IN-AarohiNeural": "mr-IN-ManoharNeural",
                "en-IN-PrabhatNeural": "en-IN-NeerjaNeural", "en-IN-NeerjaNeural": "en-IN-PrabhatNeural"}

def synthesize(text, path, voice, rate="+0%", pitch="+0Hz", volume="+0%"):
    """Free Microsoft voices sometimes return no audio for a moment. Retry, then relax rate/pitch, then use the sister voice."""
    import edge_tts
    text = re.sub(r"\s+", " ", text).strip()
    tries = [(voice, rate, pitch), (voice, rate, pitch), (voice, "+0%", "+0Hz"), (BACKUP_VOICE.get(voice, voice), "+0%", "+0Hz")]
    err = None
    for k, (v, r, p) in enumerate(tries):
        try:
            async def go():
                await edge_tts.Communicate(text, voice=v, rate=r, pitch=p, volume=volume).save(path)
            run_async(go())
            if os.path.exists(path) and os.path.getsize(path) > 500:
                if k >= 2: print(f"\n  (voice service was fussy - used voice={v}, rate={r}, pitch={p} for one line)")
                return
        except Exception as e:
            err = e
        time.sleep(2 * (k + 1))
    raise RuntimeError(f"The voice service returned no audio for: '{text[:60]}...' ({type(err).__name__}). "
                       "Wait a minute and run this cell again (finished lines are kept), or pick another voice in cell 3.")

def natural_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

TRANS_CACHE = {}
def translate(text, src, dst):
    """Returns (text, ok). The free translator limits requests, so go slowly, retry, and remember results."""
    key = (src, dst, text)
    if key in TRANS_CACHE: return TRANS_CACHE[key], True
    from deep_translator import GoogleTranslator
    for attempt in range(5):
        try:
            out = GoogleTranslator(source=src, target=dst).translate(text)
            if out:
                TRANS_CACHE[key] = out; time.sleep(0.8); return out, True
        except Exception:
            pass
        time.sleep(3 * (attempt + 1))
    return text, False

print("Ready. FFmpeg:", FFMPEG)

## 3. Settings - everything you can change is in this one cell

**Sentences.** One sentence per line, in the same order as your photos. Start a line with `Name:` (a name from `SPEAKERS`) to have that character say it.

**Voice.** Change `NARRATOR` (voice + `rate` speed + `pitch`). Hindi: `hi-IN-MadhurNeural` (male), `hi-IN-SwaraNeural` (female). Indian English: `en-IN-PrabhatNeural`, `en-IN-NeerjaNeural`.
To change the voice of one single sentence only, use `LINE_VOICES` (line numbers start at 1). Hear the options in cell 6.

In [ ]:
# ============================ SENTENCES (one per photo) ============================
SCRIPT = """
 I was born on November 7, 1888, in Tiruchirappalli, India, into a family that valued education and knowledge. Even as a child, I was fascinated by the world around me, and I believed that nature was hiding secrets waiting for someone curious enough to uncover them.

 While other children spent their time playing, I found myself drawn toward books, numbers, and scientific questions. I wanted to understand how the world worked, from the colors of the sky to the mysterious behavior of light, and every answer only made me more curious.

 My father was a teacher of mathematics and physics, and his influence helped nurture my interest in science. Surrounded by books and scientific ideas, I developed a habit that would shape my entire life: questioning everything and searching for evidence rather than blindly accepting what others told me.

 I was an exceptionally bright student, and my academic journey advanced remarkably quickly. I entered Presidency College in Madras at a young age, studied physics, earned my bachelor's degree in 1904, and received academic distinctions that demonstrated my extraordinary potential.

 In 1907, I completed my master's degree in physics with outstanding results. Since scientific opportunities in India were limited at the time, I joined the Indian Finance Department as an Assistant Accountant General, but deep inside, I knew that my true passion was discovering the secrets of the universe.

 Even while working in government service, I refused to abandon science. I discovered a scientific institution in Calcutta where I could conduct experiments after working hours, and I spent my free time investigating the mysteries of nature, determined to prove that my circumstances would never destroy my dreams.

 I became fascinated by the physics of sound and the secrets hidden inside musical instruments. I investigated how instruments such as violins and Indian drums produced their distinctive tones, demonstrating that mathematics and physics could explain the beauty of music and revealing the scientific principles behind sounds that had fascinated humanity for centuries.

 In 1917, I made a bold decision and left my government position to become a professor of physics at the University of Calcutta. For the first time, I could devote myself more fully to scientific research, teaching, and experimentation, and I knew that this decision would change the direction of my life forever.

 In 1921, while traveling by ship across the Mediterranean Sea, I gazed at the deep blue water and began questioning what I saw. Why did the sea appear blue? Was it simply reflecting the color of the sky, or was something more mysterious happening when sunlight interacted with water?

 That simple question refused to leave my mind. While others admired the beautiful scenery, I wondered whether the answer could be found by studying the behavior of light itself, and I became determined to investigate the physical processes responsible for the colors of the natural world.

 After returning to India, I began conducting experiments with light and different transparent substances. I carefully observed the light scattered by liquids and other materials, searching for clues that could reveal how light interacted with the tiny particles that made up matter.

My investigations required patience, precision, and countless observations. I worked with scientific instruments, optical filters, and spectroscopic techniques to examine scattered light, determined to discover whether the light emerging from a material could carry information that ordinary observation could not reveal.

 Then, on February 28, 1928, our experiments revealed something extraordinary. A small portion of the scattered light had a different wavelength from the incoming light, showing that light could exchange energy with molecules as it interacted with matter. At that moment, I realized that we had uncovered a remarkable physical phenomenon.

 I understood that this discovery could transform the way scientists investigated matter. The phenomenon was later named the Raman Effect, and it provided a new method for studying molecular structures by examining changes in the frequency of scattered light, opening a powerful new chapter in spectroscopy.

 News of the discovery spread throughout the scientific community, bringing international recognition to our research. I felt proud that important scientific progress was emerging from India, and I hoped our achievement would inspire young researchers to believe that curiosity and determination could overcome even the greatest obstacles.

 In 1930, I received the Nobel Prize in Physics for my work on the scattering of light and the discovery of the effect that bears my name. I became the first Asian and the first non-white recipient of the Nobel Prize in Physics, and the honor demonstrated that scientific achievements from India could earn recognition across the world.

 But I never believed that winning a Nobel Prize meant my scientific journey was complete. I continued exploring the mysteries of light, optics, crystals, and the properties of materials, always searching for new explanations, because I knew that every discovery revealed new questions waiting to be answered.

 In 1933, I became the director of the Indian Institute of Science in Bangalore, where I worked to strengthen scientific research in India. Later, in 1948, I established the Raman Research Institute, creating a place where researchers could investigate fundamental scientific questions and explore the secrets of the natural world.

 Throughout my career, I received many honors, including a knighthood in 1929 and the Bharat Ratna in 1954. Yet I believed that my greatest contribution was not simply a discovery or an award, but the message I could give future generations: never stop questioning, never fear experimentation, and never allow circumstances to limit your scientific imagination.

 I left this world on November 21, 1970, but my discovery continues to help scientists study materials, molecules, and substances in fields ranging from chemistry to medicine. My journey began with a simple question about the blue sea, and it became a discovery that changed science forever. My name is Chandrasekhara Venkata Raman, C. V. Raman, and this is the story of how curiosity helped me uncover one of nature's hidden secrets.
"""

SCRIPT_LANGUAGE = "en"       # language the sentences above are written in: "en" or "hi" (any code like "mr", "ta" also works for translation)
LANGUAGE        = "hi"       # language of the voice + subtitles. Same as SCRIPT_LANGUAGE = no translation (free translator is used otherwise)

# ============================ VOICES ============================
NARRATOR = dict(voice="hi-IN-MadhurNeural", rate="-10%", pitch="-20Hz")
SPEAKERS = {                 # name written before the colon  ->  voice style
    "Mother":      dict(voice="hi-IN-SwaraNeural",  rate="+0%",  pitch="+0Hz"),
    "Grandmother": dict(voice="hi-IN-SwaraNeural",  rate="-12%", pitch="-25Hz"),
    "Father":      dict(voice="hi-IN-MadhurNeural", rate="-5%",  pitch="-12Hz"),
    "Child":       dict(voice="hi-IN-SwaraNeural",  rate="+8%",  pitch="+30Hz"),
}
LINE_VOICES = {              # optional: change the voice of one line. Example:  3: dict(voice="hi-IN-SwaraNeural", rate="-5%", pitch="+0Hz")
}

# ============================ TIMING ============================
PAUSE_AFTER_LINE   = 0.6     # seconds of silence after each sentence
MIN_SHOT_SECONDS   = 3.0     # a sentence stays on screen at least this long, even when it is very short
IMAGES_PER_SCENE   = 1       # pictures used for ONE sentence.  1 = one picture per sentence.
                             # 2 or 3 = every sentence gets that many pictures in a row (taken from your upload in order),
                             #          and the sentence's time is shared between them.
                             # Different for each sentence? Use a list, one number per sentence, e.g. [2, 3, 1, 2]
                             #          (sentences missing from the list use 1).
MIN_IMAGE_SECONDS  = 2.5     # every picture stays at least this long (a sentence is stretched if it has many pictures)

# ============================ PHOTO FRAMING ============================
FIT = "blur"                 # "blur" = whole photo visible, blurred copy fills the sides (best for phone/portrait photos)
                             # "crop" = photo fills the whole 16:9 frame (edges are cut)

# ============================ MUSIC AND SOUND ============================
BGM_MODE    = "calm"         # "none" | "calm" | "happy" | "mystery" | "upload"   (calm/happy/mystery are generated here)
BGM_PATH    = None           # for "upload": path of a music file you have the right to use
BGM_VOLUME  = 0.14
BGM_DUCKING = True           # music gets quieter automatically while someone speaks
SFX_WHOOSH  = False
SFX_VOLUME  = 0.25

# ============================ MOTION ============================
TRANSITION_STYLE   = "mix"   # "mix" | "fade" | "slide_left" | "slide_right" | "wipe" | "zoom" | "dip"
TRANSITION_SECONDS = 0.9
HANDHELD           = False
DUST               = True
RAY_STRENGTH       = 10      # 0 = off
VIGNETTE           = 0.18    # 0 = off
SEED               = 7

# ============================ SUBTITLES ============================
SUBTITLES         = True
SUBTITLE_LANGUAGE = "narration"   # "narration" (same as the voice) | "english"
WORDS_PER_LINE    = 8
SUB_FONT_SIZE     = 16

for _n, _s in [("NARRATOR", NARRATOR)] + list(SPEAKERS.items()):
    if not _s["voice"].startswith(LANGUAGE + "-"):
        print(f"Note: {_n} uses voice {_s['voice']} but LANGUAGE is '{LANGUAGE}'. Use a matching voice for clear speech.")
print("Settings loaded.")

## 4. Upload your ZIP / PDF / photos

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tif", ".tiff")
INPUT_PATH = None        # None = a file picker opens (choose a .zip, a .pdf, or several photos).
                         # Or write a path / list of paths, e.g. "/content/story.pdf" or ["/content/photos.zip"]. A folder also works.
PDF_LONG_SIDE = 1800     # sharpness of PDF pages (pixels on the longer side)

def pdf_pages(path):
    import fitz   # PyMuPDF
    out = []
    doc = fitz.open(path)
    for page in doc:
        r = page.rect; sc = PDF_LONG_SIDE / max(r.width, r.height)
        pix = page.get_pixmap(matrix=fitz.Matrix(sc, sc), alpha=False)
        out.append(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
    doc.close()
    return out

def collect_files(sources):
    shutil.rmtree(PHOTO_DIR, ignore_errors=True); os.makedirs(PHOTO_DIR)
    found = []
    for p in sources:
        if os.path.isdir(p):
            root = p
        elif p.lower().endswith(".zip"):
            root = os.path.join(PHOTO_DIR, os.path.splitext(os.path.basename(p))[0])
            with zipfile.ZipFile(p) as z: z.extractall(root)
        else:
            found.append((os.path.basename(p), p)); continue
        for dp, dn, fn in os.walk(root):
            if "__MACOSX" in dp: continue
            for f in fn:
                if not f.startswith("."): found.append((os.path.relpath(os.path.join(dp, f), root), os.path.join(dp, f)))
    return found

def load_photos(sources):
    if isinstance(sources, str): sources = [sources]
    files = [(n, p) for n, p in collect_files(sources) if n.lower().endswith(IMG_EXT + (".pdf",))]
    files.sort(key=lambda x: natural_key(x[0]))
    names, imgs = [], []
    for n, p in files:
        if n.lower().endswith(".pdf"):
            for k, im in enumerate(pdf_pages(p), 1):
                names.append(f"{n} - page {k}"); imgs.append(im)
        else:
            names.append(n); imgs.append(ImageOps.exif_transpose(Image.open(p)).convert("RGB"))
    if not imgs: raise RuntimeError("No photos or PDF pages found. Upload a .zip, a .pdf, or image files.")
    return names, imgs

if INPUT_PATH is None:
    from google.colab import files
    up = files.upload()
    INPUT_PATH = [n for n in up if n.lower().endswith(IMG_EXT + (".zip", ".pdf"))]
    if not INPUT_PATH: raise RuntimeError("Please upload a .zip, a .pdf, or photos.")

PHOTO_NAMES, PHOTOS = load_photos(INPUT_PATH)
print(len(PHOTOS), "photos / PDF pages, in this order (write one sentence for each):")
for i, n in enumerate(PHOTO_NAMES, 1): print(f"  {i:>2}. {n}")

## 5. Match sentences to photos (translation + voices)

In [ ]:
def parse_speaker(p):
    for name in SPEAKERS:
        m = re.match(rf"^\s*{re.escape(name)}\s*:\s*(.+)$", p, re.I | re.S)
        if m: return name, m.group(1).strip()
    return None, p

LINES = [l.strip() for l in SCRIPT.strip().splitlines() if l.strip()]
if not LINES: raise RuntimeError("SCRIPT is empty - write one sentence per line in cell 3.")

# ---- which pictures belong to which sentence ----
if isinstance(IMAGES_PER_SCENE, (list, tuple)):
    COUNTS = [max(1, int(c)) for c in IMAGES_PER_SCENE][:len(LINES)]
    COUNTS += [1] * (len(LINES) - len(COUNTS))
else:
    COUNTS = [max(1, int(IMAGES_PER_SCENE))] * len(LINES)
GROUPS, pos = [], 0
for c in COUNTS:
    if pos >= len(PHOTOS): break
    GROUPS.append(list(range(pos, min(pos + c, len(PHOTOS))))); pos += c
need = sum(COUNTS)
if len(GROUPS) < len(LINES):
    print(f"WARNING: {len(LINES)} sentences need {need} pictures but you uploaded {len(PHOTOS)}. Only the first {len(GROUPS)} sentences will be used.")
elif len(PHOTOS) > need:
    print(f"NOTE: {len(PHOTOS) - need} extra picture(s) at the end are not used.")
elif len(GROUPS[-1]) < COUNTS[len(GROUPS) - 1]:
    print("NOTE: the last sentence has fewer pictures than asked for.")
N = len(GROUPS)

FALLBACK_VOICE = {"en": "en-IN-PrabhatNeural", "hi": "hi-IN-MadhurNeural", "mr": "mr-IN-ManoharNeural"}
PARTS, FAILED = [], []
for i in range(N):
    spk, text = parse_speaker(LINES[i])
    en = text if SCRIPT_LANGUAGE == "en" else translate(text, SCRIPT_LANGUAGE, "en")[0]
    if SCRIPT_LANGUAGE == LANGUAGE: spoken, ok = text, True
    else:                           spoken, ok = translate(text, SCRIPT_LANGUAGE, LANGUAGE)
    style = dict(LINE_VOICES.get(i + 1) or (SPEAKERS[spk] if spk else NARRATOR))
    if not ok:
        FAILED.append(i + 1)
        style["voice"] = FALLBACK_VOICE.get(SCRIPT_LANGUAGE, style["voice"])   # untranslated text needs a voice of its own language
    PARTS.append(dict(speaker=spk, text=text, en=en, spoken=spoken, imgs=GROUPS[i], **style))
    print(f"{i+1:>2}. [{', '.join(PHOTO_NAMES[j] for j in GROUPS[i])}]  {spk or 'narrator'}: {spoken[:50]}" + ("   <-- NOT translated" if not ok else ""))
if FAILED:
    print(f"\nTranslation failed for line(s) {FAILED}: the free translator is busy. Run THIS cell again in a minute - "
          "lines that worked are remembered, only the failed ones are retried. (If you go on now, those lines are spoken untranslated.)")

## 6. Hear the voices *(optional)*
Plays a sample of each voice so you can choose `NARRATOR` in cell 3.

In [ ]:
from IPython.display import Audio, display

VOICES_TO_TRY = {
    "Hindi male (Madhur)":          ("hi-IN-MadhurNeural", "+0%",  "+0Hz"),
    "Hindi male, deeper & slower":  ("hi-IN-MadhurNeural", "-8%",  "-12Hz"),
    "Hindi female (Swara)":         ("hi-IN-SwaraNeural",  "+0%",  "+0Hz"),
    "Hindi female, older voice":    ("hi-IN-SwaraNeural",  "-12%", "-25Hz"),
    "Hindi child-like":             ("hi-IN-SwaraNeural",  "+8%",  "+30Hz"),
    "Marathi female (Aarohi)":      ("mr-IN-AarohiNeural", "+0%",  "+0Hz"),
    "Marathi male (Manohar)":       ("mr-IN-ManoharNeural","+0%",  "+0Hz"),
    "Indian English female":        ("en-IN-NeerjaNeural", "+0%",  "+0Hz"),
    "Indian English male":          ("en-IN-PrabhatNeural","+0%",  "+0Hz"),
    "US English male (Guy)":        ("en-US-GuyNeural",    "+0%",  "+0Hz"),
    "UK English male (Ryan)":       ("en-GB-RyanNeural",   "+0%",  "+0Hz"),
}
SAMPLE = {"hi": "\u092e\u0948\u0902 \u0906\u092a\u0915\u094b \u0905\u092a\u0928\u0947 \u0917\u093e\u0901\u0935 \u0915\u0940 \u090f\u0915 \u0938\u091a\u094d\u091a\u0940 \u0915\u0939\u093e\u0928\u0940 \u0938\u0941\u0928\u093e\u0928\u0947 \u091c\u093e \u0930\u0939\u093e \u0939\u0942\u0901\u0964",
          "mr": "\u092e\u0940 \u0924\u0941\u092e\u094d\u0939\u093e\u0932\u093e \u092e\u093e\u091d\u094d\u092f\u093e \u0917\u093e\u0935\u093e\u0924\u0940\u0932 \u090f\u0915 \u0916\u0930\u0940 \u0917\u094b\u0937\u094d\u091f \u0938\u093e\u0902\u0917\u0924\u094b.",
          "en": "I am going to tell you a true story from my village."}

for label, (v, rate, pitch) in VOICES_TO_TRY.items():
    lang = v.split("-")[0]
    path_ = os.path.join(OUTPUT_DIR, "voice_" + hashlib.md5(label.encode()).hexdigest()[:6] + ".mp3")
    try:
        synthesize(SAMPLE.get(lang, SAMPLE["en"]), path_, v, rate, pitch)
        print(f"{label}   ->   voice='{v}', rate='{rate}', pitch='{pitch}'")
        display(Audio(path_))
    except Exception as e:
        print(label, "- could not generate:", str(e)[:80])

## 7. Animation engine - camera moves, transitions, ambient effects

In [ ]:
SUPER = 1.25                                   # working picture is 125% of the frame, so the camera has room to move
BW, BH = int(W * SUPER), int(H * SUPER)

def smoothstep(x):
    x = np.clip(x, 0.0, 1.0)
    return x * x * (3 - 2 * x)

def cam_ease(u):
    if u < 0:  return 0.5 * u
    if u > 1:  return 1 + 0.5 * (u - 1)
    return 0.5 * u + 0.5 * float(smoothstep(u))

def make_base(img):
    """Photo -> working picture of BW x BH. FIT='blur': whole photo + blurred fill. FIT='crop': centre crop."""
    r = max(BW / img.width, BH / img.height)
    cover = img.resize((max(BW, round(img.width * r)), max(BH, round(img.height * r))), Image.LANCZOS)
    x0, y0 = (cover.width - BW) // 2, (cover.height - BH) // 2
    cover = cover.crop((x0, y0, x0 + BW, y0 + BH))
    if FIT != "blur":
        return cover
    bg = cover.resize((BW // 8, BH // 8), Image.BILINEAR).filter(ImageFilter.GaussianBlur(3)).resize((BW, BH), Image.BILINEAR)
    bg = Image.fromarray((np.asarray(bg).astype(np.float32) * 0.65).astype(np.uint8))
    s = min(BW / img.width, BH / img.height)
    fg = img.resize((max(1, round(img.width * s)), max(1, round(img.height * s))), Image.LANCZOS)
    bg.paste(fg, ((BW - fg.width) // 2, (BH - fg.height) // 2))
    return bg

MOTIONS = ["zoom_in", "pan_right", "zoom_out", "pan_left", "push_focus", "pan_up", "drift_in", "pan_down"]

def camera_params(motion, u, focus):
    e = cam_ease(u); z, cx, cy = 1.1, 0.5, 0.5
    if   motion == "zoom_in":    z = 1.02 + 0.16 * e
    elif motion == "zoom_out":   z = 1.18 - 0.16 * e
    elif motion == "pan_right":  z = 1.18; cx = 0.43 + 0.14 * e
    elif motion == "pan_left":   z = 1.18; cx = 0.57 - 0.14 * e
    elif motion == "pan_up":     z = 1.18; cy = 0.57 - 0.14 * e
    elif motion == "pan_down":   z = 1.18; cy = 0.43 + 0.14 * e
    elif motion == "push_focus": z = 1.02 + 0.18 * e; cx = 0.5 + (focus[0] - 0.5) * 0.9 * e; cy = 0.5 + (focus[1] - 0.5) * 0.9 * e
    elif motion == "drift_in":   z = 1.04 + 0.12 * e; cx = 0.47 + 0.06 * e; cy = 0.53 - 0.05 * e
    return z, cx, cy

def shot_frame(shot, t):
    u = (t - shot["s"]) / max(shot["e"] - shot["s"], 1e-6)
    z, cx, cy = camera_params(shot["motion"], u, shot["focus"])
    if HANDHELD:
        ph = shot["focus"][0] * 10
        cx += 0.0012 * math.sin(t * 3.1 + ph) + 0.0007 * math.sin(t * 5.3 + 2 * ph)
        cy += 0.0010 * math.sin(t * 2.7 + ph) + 0.0006 * math.sin(t * 6.1 + ph)
    z = max(1.0, min(z, 1.3))
    bw, bh = BW / z, BH / z
    x0 = min(max(cx * BW - bw / 2, 0), BW - bw); y0 = min(max(cy * BH - bh / 2, 0), BH - bh)
    return np.asarray(BASES[shot["img"]].resize((W, H), Image.BILINEAR, box=(x0, y0, x0 + bw, y0 + bh)))

def _zoom_about_center(frame, s):
    bw, bh = W / s, H / s
    x0, y0 = (W - bw) / 2, (H - bh) / 2
    return np.asarray(Image.fromarray(frame).resize((W, H), Image.BILINEAR, box=(x0, y0, x0 + bw, y0 + bh)))

def transition(kind, A, B, p):
    e = float(smoothstep(p))
    if kind == "fade":
        return ((1 - e) * A.astype(np.float32) + e * B.astype(np.float32)).astype(np.uint8)
    if kind == "dip":
        if e < 0.5: return (A.astype(np.float32) * (1 - 2 * e)).astype(np.uint8)
        return (B.astype(np.float32) * (2 * e - 1)).astype(np.uint8)
    if kind in ("slide_left", "slide_right"):
        s = int(round(e * W)); out = np.empty_like(A)
        if kind == "slide_left":
            out[:, :W - s] = A[:, s:]; out[:, W - s:] = B[:, :s]
        else:
            out[:, s:] = A[:, :W - s]; out[:, :s] = B[:, W - s:]
        return out
    if kind == "wipe":
        soft = 140.0; pos = e * (W + soft) - soft
        m = np.clip((pos + soft - np.arange(W, dtype=np.float32)) / soft, 0, 1)[None, :, None]
        return (A.astype(np.float32) * (1 - m) + B.astype(np.float32) * m).astype(np.uint8)
    if kind == "zoom":
        Az = _zoom_about_center(A, 1 + 0.35 * e); Bz = _zoom_about_center(B, 1.35 - 0.35 * e)
        return ((1 - e) * Az.astype(np.float32) + e * Bz.astype(np.float32)).astype(np.uint8)
    return A if p < 0.5 else B

TRANSITION_CYCLE = ["fade", "zoom", "slide_left", "wipe", "fade", "slide_right", "dip", "zoom"]

# ---------------- ambient effects ----------------
_yy, _xx = np.mgrid[0:H, 0:W].astype(np.float32)
_rayfade = np.clip(1.15 - (_xx / W) * 0.9 - (_yy / H) * 0.5, 0, 1)
_BAND = 55.0
_OX, _OY = int(0.8 * _BAND * 2 * math.pi) + 2, int(0.6 * _BAND * 2 * math.pi) + 2
_ty, _tx = np.mgrid[-_OY:H, -_OX:W].astype(np.float32)
_RAYTEX = (np.clip(np.sin((0.8 * _tx + 0.6 * _ty) / _BAND) * 0.5 + 0.5, 0, 1) ** 3).astype(np.float32)
_TINT = np.array([1.0, 0.95, 0.8], dtype=np.float32)
_ray_cache = {}

_vig = None
def _vignette():
    global _vig
    if _vig is None:
        d = np.sqrt(((_xx - W / 2) / (W / 2)) ** 2 + ((_yy - H / 2) / (H / 2)) ** 2)
        _vig = (1 - VIGNETTE * np.clip(d - 0.55, 0, 1) ** 1.5)[:, :, None].astype(np.float32)
    return _vig

def _ray_tex(t):
    phase = (0.5 * t) % (2 * math.pi)
    ox, oy = int(0.8 * _BAND * phase), int(0.6 * _BAND * phase)
    return _RAYTEX[_OY - oy:_OY - oy + H, _OX - ox:_OX - ox + W]

def _ray_weights():
    if RAY_STRENGTH not in _ray_cache:
        _ray_cache.clear()
        _ray_cache[RAY_STRENGTH] = ((_rayfade * RAY_STRENGTH)[:, :, None] * _TINT).astype(np.float32)
    return _ray_cache[RAY_STRENGTH]

_rng = np.random.default_rng(11)
N_DUST = 48
_dust = dict(x=_rng.uniform(0, W, N_DUST), y=_rng.uniform(0, H, N_DUST), vx=_rng.uniform(-8, 14, N_DUST),
             vy=_rng.uniform(-14, -3, N_DUST), ph=_rng.uniform(0, 6.28, N_DUST), r=_rng.integers(2, 5, N_DUST))
def _sprite(r):
    yy, xx = np.mgrid[-r*2:r*2+1, -r*2:r*2+1].astype(np.float32)
    return np.exp(-(xx**2 + yy**2) / (2 * (r * 0.9) ** 2))
_sprites = {r: _sprite(int(r)).astype(np.float32) for r in range(2, 6)}

_fbuf = np.empty((H, W, 3), np.float32)
_gbuf = np.empty((H, W, 3), np.float32)

def add_effects(frame, t):
    if not (DUST or RAY_STRENGTH or VIGNETTE):
        return frame
    f = _fbuf
    if VIGNETTE: np.multiply(frame, _vignette(), out=f)
    else:        f[...] = frame
    if RAY_STRENGTH:
        np.multiply(_ray_tex(t)[:, :, None], _ray_weights(), out=_gbuf)
        f += _gbuf
    if DUST:
        for i in range(N_DUST):
            x = int((_dust["x"][i] + _dust["vx"][i] * t + 10 * math.sin(t * 0.6 + _dust["ph"][i])) % W)
            y = int((_dust["y"][i] + _dust["vy"][i] * t) % H)
            sp = _sprites[int(_dust["r"][i])]; h = sp.shape[0] // 2
            tw = 0.55 + 0.45 * math.sin(t * 1.7 + _dust["ph"][i])
            y0, y1, x0, x1 = max(y - h, 0), min(y + h + 1, H), max(x - h, 0), min(x + h + 1, W)
            if y1 > y0 and x1 > x0:
                f[y0:y1, x0:x1] += (sp[y0 - (y - h):y1 - (y - h), x0 - (x - h):x1 - (x - h)] * 110 * tw)[:, :, None] * _TINT
    np.minimum(f, 255, out=f)
    return f.astype(np.uint8)

BASES = {}                                     # picture number -> prepared picture
print("Animation engine ready.")

## 8. Audio: music, whoosh sounds, mixing

In [ ]:
SR = 24000

def _midi(m): return 440.0 * 2 ** ((m - 69) / 12)

MOODS = {
    "calm":    dict(chords=[[57, 60, 64], [53, 57, 60], [48, 52, 55], [55, 59, 62]], chord_sec=8.0, steps=8,  arp=0.45, pad=0.55, bass=0.30, arp_vol=0.16),
    "happy":   dict(chords=[[60, 64, 67], [55, 59, 62], [57, 60, 64], [53, 57, 60]], chord_sec=4.0, steps=8,  arp=0.95, pad=0.35, bass=0.30, arp_vol=0.22),
    "mystery": dict(chords=[[57, 60, 64], [50, 53, 57], [52, 56, 59], [57, 60, 64]], chord_sec=8.0, steps=4,  arp=0.50, pad=0.55, bass=0.40, arp_vol=0.14),
}

def synth_bgm(mood):
    m = MOODS[mood]; cs = m["chord_sec"]; total = cs * len(m["chords"])
    n = int(SR * total); out = np.zeros(n); rng = np.random.default_rng(3)
    for ci, chord in enumerate(m["chords"]):
        t0 = ci * cs; i0 = int(t0 * SR); i1 = min(int((t0 + cs + 1.0) * SR), n)
        tt = np.arange(i1 - i0) / SR
        env = np.minimum(tt / 1.2, 1.0) * np.where(tt < cs, 1.0, np.clip(1 - (tt - cs) / 1.0, 0, 1))
        for midi in chord:
            f = _midi(midi)
            out[i0:i1] += m["pad"] * env * (np.sin(2*np.pi*f*tt) + 0.6*np.sin(2*np.pi*f*1.003*tt) + 0.25*np.sin(2*np.pi*2*f*tt)) / 3
        out[i0:i1] += m["bass"] * env * np.sin(2*np.pi*_midi(chord[0] - 12) * tt)
        step = cs / m["steps"]
        for s in range(m["steps"]):
            if rng.random() > m["arp"]: continue
            midi = chord[s % len(chord)] + 12 * (1 + (s % 2))
            a = int((t0 + s * step) * SR); b = min(a + int(0.9 * SR), n)
            if b <= a: continue
            tt2 = np.arange(b - a) / SR
            out[a:b] += m["arp_vol"] * np.exp(-tt2 * 4.5) * np.sin(2*np.pi*_midi(midi) * tt2) * (1 - np.exp(-tt2 * 80))
    fade = int(0.3 * SR); out[:fade] *= np.linspace(0, 1, fade); out[-fade:] *= np.linspace(1, 0, fade)
    out = out / max(np.abs(out).max(), 1e-6) * 0.6
    return (out * 32767).astype(np.int16)

def synth_whoosh(dur=0.7):
    n = int(SR * dur); noise = np.random.default_rng(5).standard_normal(n)
    a = np.linspace(0.04, 0.35, n); y = np.zeros(n); prev = 0.0
    for i in range(n):
        prev += a[i] * (noise[i] - prev); y[i] = prev
    y *= np.sin(np.pi * np.linspace(0, 1, n)) ** 2
    return y / (np.abs(y).max() + 1e-9) * 0.6

def write_wav(path, samples):
    with wave.open(path, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR)
        w.writeframes(np.asarray(samples, dtype=np.int16).tobytes())

def mix_audio(voice_wav, total, boundary_times, out_wav):
    inputs, parts, last, idx = ["-i", voice_wav], [], "[0:a]", 1
    bgm = None
    if BGM_MODE in MOODS:
        bgm = os.path.join(OUTPUT_DIR, f"_bgm_{BGM_MODE}.wav")
        if not os.path.exists(bgm): write_wav(bgm, synth_bgm(BGM_MODE))
    elif BGM_MODE == "upload" and BGM_PATH:
        bgm = BGM_PATH
    if bgm:
        inputs += ["-stream_loop", "-1", "-i", bgm]
        if BGM_DUCKING:
            parts += ["[0:a]asplit=2[v1][v2]", f"[{idx}:a]volume={BGM_VOLUME}[b]",
                      "[b][v1]sidechaincompress=threshold=0.03:ratio=8:attack=20:release=400[bd]",
                      "[v2][bd]amix=inputs=2:duration=first:dropout_transition=0,volume=2[m1]"]
        else:
            parts += [f"[{idx}:a]volume={BGM_VOLUME}[b]", "[0:a][b]amix=inputs=2:duration=first:dropout_transition=0,volume=2[m1]"]
        last = "[m1]"; idx += 1
    if SFX_WHOOSH and boundary_times:
        track = np.zeros(int(total * SR) + SR); wh = synth_whoosh()
        for tb in boundary_times:
            a = max(int((tb - 0.35) * SR), 0); b = min(a + len(wh), len(track))
            track[a:b] += wh[:b - a]
        sfx = os.path.join(OUTPUT_DIR, "_sfx.wav"); write_wav(sfx, np.clip(track, -1, 1) * 32767)
        inputs += ["-i", sfx]
        parts += [f"[{idx}:a]volume={SFX_VOLUME}[s]", f"{last}[s]amix=inputs=2:duration=first:dropout_transition=0,volume=2[m2]"]
        last = "[m2]"
    if not parts:
        shutil.copy(voice_wav, out_wav); return out_wav
    subprocess.run([FFMPEG, "-y"] + inputs + ["-filter_complex", ";".join(parts), "-map", last, "-t", f"{total:.3f}",
                   "-ar", str(SR), "-ac", "1", out_wav], check=True, capture_output=True)
    return out_wav

print("Audio tools ready. Music mode:", BGM_MODE)

## 9. Video builder

In [ ]:
def _srt_time(t):
    ms = int(round(t * 1000)); h, ms = divmod(ms, 3600000); m, ms = divmod(ms, 60000); s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def _chunks(text):
    out = []
    for sent in re.split(r"(?<=[.!?\u0964])\s+", text.strip()):
        words = sent.split()
        if not words: continue
        n = math.ceil(len(words) / WORDS_PER_LINE); size = math.ceil(len(words) / n)
        out += [" ".join(words[i:i + size]) for i in range(0, len(words), size)]
    return out

def _subtitle_filter(srt_path, texts):
    needs_dev = any("\u0900" <= ch <= "\u097F" for t in texts for ch in t)
    font, fdir = "DejaVu Sans", "/usr/share/fonts/truetype/dejavu"
    if needs_dev:
        noto = "/usr/share/fonts/truetype/noto"
        if not any("Devanagari" in f for f in (os.listdir(noto) if os.path.isdir(noto) else [])):
            subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-noto-core"], capture_output=True)
        font, fdir = "Noto Sans Devanagari", noto
    style = (f"FontName={font},FontSize={SUB_FONT_SIZE},Bold=1,PrimaryColour=&H00FFFFFF,OutlineColour=&H00000000,"
             f"BorderStyle=1,Outline=2.5,Shadow=1,Alignment=2,MarginV=18")
    vf = f"subtitles={srt_path}:fontsdir={fdir}:force_style='{style}'"
    test = subprocess.run([FFMPEG, "-y", "-f", "lavfi", "-i", "color=c=black:s=320x180:d=0.2", "-vf", vf,
                           "-frames:v", "1", "-f", "null", "-"], capture_output=True)
    return vf if test.returncode == 0 else None

def build_video(name="photo_story", preview_seconds=None):
    n_total = len(PARTS)

    # ---- 1) voice: one clip per sentence (cached); each photo lasts as long as its sentence ----
    clips, starts, speech, elapsed = [], [], [], 0.0
    for i, part in enumerate(PARTS):
        key = hashlib.md5(f"{part['voice']}|{part['rate']}|{part['pitch']}|{part['spoken']}".encode()).hexdigest()[:12]
        mp3 = os.path.join(CACHE_DIR, key + ".mp3")
        if not os.path.exists(mp3):
            synthesize(part["spoken"], mp3, part["voice"], part["rate"], part["pitch"])
        sp = get_duration(mp3); slot = max(sp + PAUSE_AFTER_LINE, MIN_SHOT_SECONDS, MIN_IMAGE_SECONDS * len(part["imgs"]))
        starts.append(elapsed); speech.append(sp); clips.append((mp3, slot)); elapsed += slot
        print(f"\r  voice {i+1}/{n_total} ({elapsed:5.0f}s)", end="")
        if preview_seconds and elapsed >= preview_seconds: break
    total, done = elapsed, len(clips)
    print(f"\nVideo length: {total:.0f}s ({total/60:.1f} min)" + ("  [preview]" if done < n_total else ""))

    # ---- 2) shots: each sentence = one or more pictures sharing its time ----
    BASES.clear()
    rng = np.random.default_rng(SEED); shots = []; mi = 0
    for i in range(done):
        idxs, slot = PARTS[i]["imgs"], clips[i][1]; n = len(idxs)
        for j, pi in enumerate(idxs):
            if pi not in BASES: BASES[pi] = make_base(PHOTOS[pi])
            shots.append(dict(img=pi, scene=i, s=starts[i] + slot * j / n, e=starts[i] + slot * (j + 1) / n,
                              motion=MOTIONS[mi % len(MOTIONS)],
                              focus=(float(rng.uniform(0.35, 0.65)), float(rng.uniform(0.35, 0.65)))))
            mi += 1
    trans, tcount = [], 0
    for k in range(len(shots) - 1):
        a, b = shots[k], shots[k + 1]
        same = a["scene"] == b["scene"]                      # next picture of the same sentence: gentle dissolve
        T = min(0.6 if same else TRANSITION_SECONDS, 0.45 * min(a["e"] - a["s"], b["e"] - b["s"]))
        if T < 0.15: kind, T = "cut", 0.0
        elif same:   kind = "fade"
        else:
            kind = TRANSITION_CYCLE[tcount % len(TRANSITION_CYCLE)] if TRANSITION_STYLE == "mix" else TRANSITION_STYLE
            tcount += 1
        trans.append(dict(kind=kind, T=T, b=a["e"], diff=not same))
    print(f"{done} sentences, {len(shots)} pictures, {len(trans)} transitions")

    # ---- 3) audio ----
    wavs = []
    for i, (mp3, slot) in enumerate(clips):
        wav = os.path.join(OUTPUT_DIR, f"_a{i:03d}.wav")
        subprocess.run([FFMPEG, "-y", "-i", mp3, "-af", "apad", "-t", f"{slot:.3f}", "-ar", str(SR), "-ac", "1", wav],
                       check=True, capture_output=True)
        wavs.append(wav)
    lst = os.path.join(OUTPUT_DIR, "_audio.txt"); open(lst, "w").write("\n".join(f"file '{w}'" for w in wavs))
    voice_wav = os.path.join(OUTPUT_DIR, "_voice.wav")
    subprocess.run([FFMPEG, "-y", "-f", "concat", "-safe", "0", "-i", lst, "-c", "copy", voice_wav], check=True, capture_output=True)
    audio = mix_audio(voice_wav, total, [tr["b"] for tr in trans if tr["diff"] and tr["kind"] != "cut"], os.path.join(OUTPUT_DIR, "_mix.wav"))

    # ---- 4) subtitles ----
    srt_path, vf = os.path.join(OUTPUT_DIR, f"{name}.srt"), None
    if SUBTITLES:
        entries = []
        for i in range(done):
            txt = PARTS[i]["spoken"] if SUBTITLE_LANGUAGE == "narration" else PARTS[i]["en"]
            chunks = _chunks(txt); wts = [len(c) for c in chunks]; t0 = starts[i]
            for c, w in zip(chunks, wts):
                seg = speech[i] * w / sum(wts); entries.append((t0, t0 + seg, c)); t0 += seg
        with open(srt_path, "w", encoding="utf-8") as f:
            for i, (a, b, c) in enumerate(entries, 1):
                f.write(f"{i}\n{_srt_time(a)} --> {_srt_time(min(b, total))}\n{c}\n\n")
        vf = _subtitle_filter(srt_path, [c for _, _, c in entries])
        print(f"{len(entries)} subtitle lines" + ("" if vf else " (this FFmpeg cannot burn them in; the .srt file is saved instead)"))

    # ---- 5) render ----
    final = os.path.join(OUTPUT_DIR, f"{name}.mp4")
    cmd = [FFMPEG, "-y", "-f", "rawvideo", "-vcodec", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}",
           "-r", str(FPS), "-i", "-", "-i", audio] + (["-vf", vf] if vf else []) + \
          ["-c:v", "libx264", "-preset", "veryfast", "-crf", "22", "-pix_fmt", "yuv420p",
           "-c:a", "aac", "-b:a", "128k", "-shortest", "-movflags", "+faststart", final]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    try:
        k = 0
        for f_i in range(int(total * FPS)):
            t = f_i / FPS
            while k < len(trans) and t >= trans[k]["b"] + trans[k]["T"] / 2:
                k += 1
            if k < len(trans) and trans[k]["T"] > 0 and t >= trans[k]["b"] - trans[k]["T"] / 2:
                T = trans[k]["T"]; p = (t - (trans[k]["b"] - T / 2)) / T
                frame = transition(trans[k]["kind"], shot_frame(shots[k], t), shot_frame(shots[k + 1], t), p)
            else:
                frame = shot_frame(shots[min(k, len(shots) - 1)], t)
            proc.stdin.write(add_effects(frame, t).tobytes())
            if f_i % (FPS * 10) == 0:
                print(f"\r  rendering {t:5.0f}s / {total:.0f}s", end="")
        proc.stdin.close()
        err = proc.stderr.read().decode(errors="replace")
        if proc.wait() != 0: raise RuntimeError(err[-3000:])
    except Exception:
        if proc.poll() is None: proc.kill()
        raise
    for p in wavs + [lst, voice_wav, os.path.join(OUTPUT_DIR, "_mix.wav"), os.path.join(OUTPUT_DIR, "_sfx.wav")]:
        if os.path.exists(p): os.remove(p)
    print("\nDone:", final)
    return final

print("Builder ready.")

## 10. Make the video
Tip: set `PREVIEW_SECONDS = 20` first to check voice and look quickly, then `None` for the full video.

In [ ]:
from IPython.display import Video, display

PREVIEW_SECONDS = None      # e.g. 20 for a quick test, None = the full video

path = build_video(preview_seconds=PREVIEW_SECONDS)
print("Length:", round(get_duration(path)), "s | Size:", round(os.path.getsize(path) / 1024**2, 1), "MB")
display(Video(path, embed=(os.path.getsize(path) < 40 * 1024**2), width=720))

### Download

In [ ]:
from google.colab import files
files.download(path)